In [12]:
! pip install chromadb sentence-transformers python-dotenv

In [13]:
import json
import os
import kagglehub
import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR NOTEBOOK.
seiftarek158_contract_nli_path = kagglehub.dataset_download('seiftarek158/contract-nli')
print('Data source import complete.')

100%|██████████| 29.1M/29.1M [00:01<00:00, 17.6MB/s]

Extracting files...


Data source import complete.


In [3]:
DATA_DIR = seiftarek158_contract_nli_path

with open(f'{DATA_DIR}/contract-nli/train.json') as f:
    train_data = json.load(f)

print('Top-level keys:', list(train_data.keys()))
print('Number of documents:', len(train_data['documents']))
print('Number of hypotheses:', len(train_data['labels']))
print('\nHypothesis IDs and texts:')
for h_id, h_info in train_data['labels'].items():
    print(f'  {h_id}: {h_info}')

Top-level keys: ['documents', 'labels']
Number of documents: 423
Number of hypotheses: 17

Hypothesis IDs and texts:
  nda-11: {'short_description': 'No reverse engineering', 'hypothesis': "Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information."}
  nda-16: {'short_description': 'Return of confidential information', 'hypothesis': 'Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.'}
  nda-15: {'short_description': 'No licensing', 'hypothesis': 'Agreement shall not grant Receiving Party any right to Confidential Information.'}
  nda-10: {'short_description': 'Confidentiality of Agreement', 'hypothesis': 'Receiving Party shall not disclose the fact that Agreement was agreed or negotiated.'}
  nda-2: {'short_description': 'None-inclusion of non-technical information', 'hypothesis': 'Confidential Information shall only include technical information.'}
  nda-1: {'short_description'

In [5]:
def populate_chunks(train_data):
    chunks = []

    for doc in train_data["documents"]:
        text = doc["text"]
        spans = doc["spans"]
        annotation_sets = doc.get("annotation_sets", [])

        # Build a lookup: span_tuple -> list of {hypothesis_id, gold_label}
        span_evidence_map = {}

        for annotation_set in annotation_sets:
            annotations = annotation_set.get("annotations", {})
            for hypothesis_id, ann_data in annotations.items():
                choice = ann_data.get("choice", "NotMentioned")
                evidence_span_indices = ann_data.get("spans", [])

                for span_index in evidence_span_indices:
                    actual_span = spans[span_index]  # resolve index -> [char_start, char_end]
                    span_key = tuple(actual_span)
                    if span_key not in span_evidence_map:
                        span_evidence_map[span_key] = []
                    span_evidence_map[span_key].append({
                        "hypothesis_id": hypothesis_id,
                        "gold_label": choice
                    })

        # Now create a Chunk for every span in the document
        for span in spans:
            char_start, char_end = span[0], span[1]
            span_text = text[char_start:char_end]
            evidence_for = span_evidence_map.get(tuple(span), [])

            chunks.append({
                "text": span_text,
                "span": [char_start, char_end],
                "evidence_for": evidence_for
            })

    return chunks


chunks = populate_chunks(train_data)
len(chunks)

32895

In [15]:
client = chromadb.CloudClient(
  api_key= os.getenv("CHROMA_API_KEY"),
  tenant='fd2eb954-04b9-4e06-adbe-b687d3c0d25b',
  database='Agentic_project'
)

In [16]:
collection = client.get_or_create_collection(
    name="nda_chunks",
    metadata={"hnsw:space": "cosine"}  # cosine similarity for text
)

# Embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")  # fast & good for semantic search

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
def store_chunks(chunks, collection, embedder, batch_size=128):
    total = len(chunks)

    for i in range(0, total, batch_size):
        batch = chunks[i : i + batch_size]

        texts     = [c["text"] for c in batch]
        embeddings = embedder.encode(texts, show_progress_bar=True).tolist()

        ids = [str(i + j) for j in range(len(batch))]  # unique ID per chunk

        # ChromaDB metadata values must be str/int/float — serialize evidence_for as JSON string
        metadatas = [
            {
                "span_start":   c["span"][0],
                "span_end":     c["span"][1],
                "evidence_for": json.dumps(c["evidence_for"])  # list → JSON string
            }
            for c in batch
        ]

        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )

        print(f"Stored {min(i + batch_size, total)}/{total} chunks") if i % 2 == 0 else None


store_chunks(chunks, collection, embedder)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Stored 128/32895 chunks


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Stored 256/32895 chunks


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

...

Stored 32640/32895 chunks


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Stored 32768/32895 chunks


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Stored 32895/32895 chunks


In [18]:
print("Total vectors stored:", collection.count())

# Peek at the first 3 stored entries
peek = collection.peek(limit=3)
for i in range(len(peek["ids"])):
    print("ID:", peek["ids"][i])
    print("Text:", peek["documents"][i][:80], "...")
    print("Metadata:", peek["metadatas"][i])
    print("---")

Total vectors stored: 32895
ID: 0
Text: NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT ...
Metadata: {'evidence_for': '[]', 'span_end': 44, 'span_start': 0}
---
ID: 1
Text: This NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT (“Agreement”) is made by and b ...
Metadata: {'span_end': 132, 'evidence_for': '[]', 'span_start': 45}
---
ID: 2
Text: (i) the Office of the United Nations High Commissioner for Refugees, having its  ...
Metadata: {'span_end': 331, 'span_start': 133, 'evidence_for': '[]'}
---


In [19]:
def query_chunks(query_text, collection, embedder, top_k=5):
    query_vector = embedder.encode([query_text]).tolist()

    results = collection.query(
        query_embeddings=query_vector,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    for i in range(len(results["ids"][0])):
        print(f"Rank {i+1}")
        print("  Text:      ", results["documents"][0][i][:100], "...")
        print("  Distance:  ", results["distances"][0][i])
        evidence = json.loads(results["metadatas"][0][i]["evidence_for"])
        print("  Evidence:  ", evidence)
        print()

query_chunks("confidentiality obligations of the receiving party", collection, embedder)

Rank 1
  Text:       The Receiving Party will instruct its Representatives who have access to the Confidential Informatio ...
  Distance:   0.15419364
  Evidence:   []

Rank 2
  Text:       The receiving Party may disclose Confidential Information: ...
  Distance:   0.15571225
  Evidence:   [{'hypothesis_id': 'nda-7', 'gold_label': 'Contradiction'}, {'hypothesis_id': 'nda-8', 'gold_label': 'Entailment'}, {'hypothesis_id': 'nda-5', 'gold_label': 'Entailment'}]

Rank 3
  Text:       In addition, the Receiving Party acknowledges that the following specific Confidential Information,  ...
  Distance:   0.1667291
  Evidence:   [{'hypothesis_id': 'nda-1', 'gold_label': 'Contradiction'}]

Rank 4
  Text:       Hence, the Receiving Party will be responsible for ensuring that the obligations of confidentiality  ...
  Distance:   0.17034197
  Evidence:   []

Rank 5
  Text:       Likewise the Party which receives the disclosed Confidential Information shall be regarded as the Re ...
  Distance:   0

In [25]:
train_data['documents'][0]

{'id': 34,
 'file_name': 'Annex E_Non-Disclosure and Confidentiality Agreement.pdf',
 'text': "NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT\nThis NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT (“Agreement”) is made by and between:\n(i) the Office of the United Nations High Commissioner for Refugees, having its headquarters located at 94 rue de Montbrillant, 1202 Geneva, Switzerland (hereinafter “UNHCR” or the “Discloser”); and\n(ii) ________________________ , a company established in accordance with the laws of ________________________ and having its principal offices located at ________________________________________________ (hereinafter the “Bidder” or the “Recipient”).\nThe Discloser and Recipient are also referred to collectively as the “Parties” and individually as a “Party”.\nRECITALS\nWHEREAS in connection with RFP/2014/620, Request for Proposal for the provision Off-the-shelf Soft-skill, IT Online and HR specific E-learning Courses (the “RFP”), it is advantageous to share certai

In [32]:
train_data['documents'][0]['annotation_sets'][0]['annotations']['nda-12']

{'choice': 'Entailment', 'spans': [30, 34]}

In [ ]:
from vector_rag_query import build_rag_prompt

test_contract = train_data['documents'][6]

prompt = build_rag_prompt(
    contract_text = test_contract['text'],
    spans         = test_contract['spans'],
    hypothesis_id = "nda-12",
    collection    = collection,
    embedder       = embedder,
)

print(prompt)

Classify the hypothesis based on the contract. Respond with ONLY valid JSON nothing else:
{"label": "ENTAILED" | "CONTRADICTED" | "NOT_MENTIONED", "evidence": ["exact quote 1", "exact quote 2"]}
If label is NOT_MENTIONED, evidence must be [].
Evidence must be copied verbatim from the contract text, word for word. Do not paraphrase or invent.
Contract:
Mutual Non-Disclosure Agreement
THIS MUTUAL NON-DISCLOSURE AGREEMENT is made on (insert date) 2012
Between
(A) Wollaston School whose address for notifications under this Agreement is at Irchester Road, Wollaston, Wellingborough, Northamptonshire, NN29 7PH and
(B) [ insert company name ] a company incorporated in [ ] (registered no. [ ]), and whose registered office is at [ ] and whose address for notifications under this Agreement is [ ] (“ “).
Whereas
Wollaston School and [ ] are respectively the owners of Confidential Information which they have agreed to disclose to each other for the Purpose on the terms and conditions set out in thi